In [3]:
import random
import string
import pandas as pd
import numpy as np

# ----------------------------------------------------------------------
# 0. Reproducibility
# ----------------------------------------------------------------------
SEED = 20240607
random.seed(SEED)
np.random.seed(SEED)

# ----------------------------------------------------------------------
# 1. Load inputs
# ----------------------------------------------------------------------
dir_df = pd.read_csv("directory.csv", dtype=str, encoding="latin-1")
geo_df = pd.read_csv("PH_Adm4_BgySubMuns.csv", dtype={"adm4_psgc": str})

dir_df["psgc"] = dir_df["10 Digit Code"].str.strip()
dir_df["Urban/Rural"] = dir_df["Urban/Rural"].str.strip().str.title()  # "Urban"/"Rural"
geo_df["psgc"] = geo_df["adm4_psgc"].str.strip().str.zfill(10)

# ----------------------------------------------------------------------
# 2. Deterministically pick 3 URBAN + 3 RURAL barangays
# ----------------------------------------------------------------------
rng = random.Random(SEED)

urban_pool = sorted(dir_df.loc[dir_df["Urban/Rural"] == "Urban", "Barangays"].unique())
rural_pool = sorted(dir_df.loc[dir_df["Urban/Rural"] == "Rural", "Barangays"].unique())

if len(urban_pool) < 3 or len(rural_pool) < 3:
    raise ValueError(
        f"Need >=3 of each. Found urban={len(urban_pool)}, rural={len(rural_pool)}"
    )

chosen_urban = rng.sample(urban_pool, 3)
chosen_rural = rng.sample(rural_pool, 3)
chosen = chosen_urban + chosen_rural

print("Chosen URBAN barangays:", chosen_urban)
print("Chosen RURAL barangays:", chosen_rural)

chosen_df = (
    dir_df[dir_df["Barangays"].isin(chosen)]
    .drop_duplicates("Barangays")
    .set_index("Barangays")
    .loc[chosen]                       # preserve urban-then-rural order
    .reset_index()
)

# ----------------------------------------------------------------------
# 3. Match to PH_Adm4_BgySubMuns.csv for geometry
# ----------------------------------------------------------------------
merged = chosen_df.merge(
    geo_df[["psgc", "adm4_en", "len_crs", "area_crs", "len_km", "area_km2"]],
    on="psgc",
    how="left",
)

if merged["len_crs"].isna().all():
    print("\n⚠️  WARNING: No PSGC matches found — geometry columns will be blank.\n")

# ----------------------------------------------------------------------
# 4. Synthetic coordinate generator (bbox around centroid)
# ----------------------------------------------------------------------
CENTROIDS = {
    "A. Bonifacio-Caguioa-Rimando (ABCR)": (14.6042, 120.9822),
    "Abanao-Zandueta-Kayong-Chugum-Otek (AZKCO)": (16.4110, 120.5960),
    "Alfonso Tabora":                       (14.6070, 120.9840),
    # add rural centroids here as needed
}
DEFAULT_CENTROID = (14.5995, 120.9842)
BBOX_DEG = 0.003

def sample_latlon(bgy):
    clat, clon = CENTROIDS.get(bgy, DEFAULT_CENTROID)
    return (
        round(random.uniform(clat - BBOX_DEG, clat + BBOX_DEG), 6),
        round(random.uniform(clon - BBOX_DEG, clon + BBOX_DEG), 6),
    )

# ----------------------------------------------------------------------
# 5. Helpers
# ----------------------------------------------------------------------
def make_id():
    chars = string.ascii_uppercase + string.digits
    return "".join(random.choice(chars) for _ in range(10))

def unique_id(existing):
    while True:
        i = make_id()
        if i not in existing:
            existing.add(i)
            return i

# ----------------------------------------------------------------------
# 6. Generate records
# ----------------------------------------------------------------------
N_PER_BGY = 8
records, used_ids = [], set()

for _, row in merged.iterrows():
    bgy = row["Barangays"]
    for _ in range(N_PER_BGY):
        beds, cr = random.randint(1, 10), random.randint(1, 10)
        kitchen, wifi, security = (random.randint(0, 1) for _ in range(3))
        amenity_sum = beds + cr + kitchen + wifi
        lat, lon = sample_latlon(bgy)

        records.append({
            "boarding_house_id": unique_id(used_ids),
            "barangay": bgy,
            "urban_rural": row["Urban/Rural"],   # ← handy audit column
            "lat": lat,
            "long": lon,
            "monthly_cost": random.randint(2500, 50000),
            "beds": beds,
            "cr": cr,
            "kitchen": kitchen,
            "wifi": wifi,
            "amenity_sum": amenity_sum,
            "amenity_ave": round(amenity_sum / 4, 2),
            "security": security,
            "len_crs": row["len_crs"],
            "area_crs": row["area_crs"],
            "len_km": row["len_km"],
            "area_km2": row["area_km2"],
        })

out = pd.DataFrame(records)
out.to_csv("synthetic_boarding_houses.csv", index=False)
print(f"\nWrote {len(out)} rows to synthetic_boarding_houses.csv")
print(out.head(10).to_string(index=False))

Chosen URBAN barangays: ['Teodora Alonzo', 'Outlook Drive', 'Session Road Area']
Chosen RURAL barangays: ['Phil-Am', 'Modern Site, West', 'Bayan Park East']

Wrote 48 rows to synthetic_boarding_houses.csv
boarding_house_id       barangay urban_rural       lat       long  monthly_cost  beds  cr  kitchen  wifi  amenity_sum  amenity_ave  security  len_crs  area_crs  len_km  area_km2
       VISB25CT89 Teodora Alonzo       Urban 14.599930 120.982604         16628    10   7        1     0           18         4.50         0     1428     70748       1       0.0
       NT6Q5HSIT5 Teodora Alonzo       Urban 14.599003 120.982759         31626     3   2        0     0            5         1.25         0     1428     70748       1       0.0
       6227PP4YY4 Teodora Alonzo       Urban 14.598539 120.983122         35061     2  10        1     1           14         3.50         0     1428     70748       1       0.0
       EEWBN4Y8N7 Teodora Alonzo       Urban 14.599276 120.985476         18934    